## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [15]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
import os

In [16]:
MODEL = "(paid) gpt-5.4" #"gpt-4.1-nano" #"gpt-6-luna"
DB_NAME = "vector_db"
load_dotenv(override=True)
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_BASE_URL = os.getenv('OPENAI_BASE_URL')

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [17]:
retriever = vectorstore.as_retriever()
llm = ChatOpenAI(temperature=0, model_name=MODEL, base_url=OPENAI_BASE_URL, api_key=OPENAI_API_KEY, reasoning_effort="none")

### These LangChain objects implement the method `invoke()`

In [12]:
retriever.invoke("Who is Avery?")

[Document(id='e138bce4-4bd8-4dc6-a40e-9cd8c70ec0c6', metadata={'source': 'knowledge-base/employees/Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

In [18]:
llm.invoke("Who is Avery?")

AIMessage(content='I don’t know which Avery you mean.\n\n“Avery” could refer to:\n- a person you know\n- a public figure\n- a fictional character\n- a company or brand\n- a given name/surname in general\n\nIf you want, give me a little context, like:\n- “Avery from *X*”\n- “Avery the singer”\n- “What does the name Avery mean?”', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 87, 'prompt_tokens': 10, 'total_tokens': 97, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': '(paid) gpt-5.4', 'system_fingerprint': None, 'id': 'chatcmpl-EjrxTTDv2eMF2gTlvGhk40r7W2eEv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--43a3cc0b-f409-489e-8118-14f03826e39c-0', usage_metadata={'input_tokens': 10, 'output_tokens': 87, 'total_t

## Time to put this together!

In [19]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [21]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [22]:
answer_question("Who is Averi Lancaster?", [])

'You may mean **Avery Lancaster**.\n\n**Avery Lancaster** is **Insurellm’s Co-Founder and Chief Executive Officer (CEO)**, based in **San Francisco, California**.\n\nA quick background:\n- **Born:** March 15, 1985\n- **Role at Insurellm:** Co-Founder & CEO since **2015**\n- Avery co-founded Insurellm and has led the company’s growth as a leading insurance technology provider.\n- Before Insurellm, Avery was a **Senior Product Manager at Innovate Insurance Solutions** from **2013 to 2015**.\n\nAvery is known for **innovative leadership** and **risk management expertise**.\n\nIf you want, I can also give you a short bio or a more detailed career summary.'

## What could possibly come next? 😂

In [ ]:
gr.ChatInterface(answer_question).launch()

## Admit it - you thought RAG would be more complicated than that!!